In [2]:
import torch
import pandas
import numpy 
import sklearn
import transformers
from transformers import XLMRobertaModel , XLMRobertaTokenizer, TrainingArguments, Trainer
import torch.nn as nn

print("✅ PyTorch:", torch.__version__)
print("✅ CUDA:", torch.cuda.is_available())
print("✅ Pandas:", pandas.__version__)
print("✅ NumPy:", numpy.__version__)
print("✅ Scikit-learn:", sklearn.__version__)
print("✅ torch:", torch.__version__)
print(f"Transformers version: {transformers.__version__}")

c:\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ PyTorch: 2.6.0+cu124
✅ CUDA: True
✅ Pandas: 3.0.1
✅ NumPy: 2.4.3
✅ Scikit-learn: 1.8.0
✅ torch: 2.6.0+cu124
Transformers version: 5.5.4


In [3]:
# 1. Vérifier si CUDA disponible
print(f"CUDA disponible: {torch.cuda.is_available()}")

# 2. Informations GPU
if torch.cuda.is_available():
    print(f"\n📊 INFORMATIONS GPU:")
    print(f"Nom: {torch.cuda.get_device_name(0)}")
    print(f"VRAM totale: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"VRAM utilisée: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"VRAM réservée: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")
    print(f"Capacité: {torch.cuda.get_device_capability(0)}")
    print(f"Multi-processeurs: {torch.cuda.get_device_properties(0).multi_processor_count}")
    
    # 3. Version CUDA
    print(f"\n🔧 VERSIONS:")
    print(f"CUDA version (PyTorch): {torch.version.cuda}")
    print(f"cuDNN version: {torch.backends.cudnn.version()}")

else:
    print("❌ Aucun GPU détecté")

CUDA disponible: True

📊 INFORMATIONS GPU:
Nom: NVIDIA GeForce RTX 3080 Ti
VRAM totale: 12.88 GB
VRAM utilisée: 0.00 GB
VRAM réservée: 0.00 GB
Capacité: (8, 6)
Multi-processeurs: 80

🔧 VERSIONS:
CUDA version (PyTorch): 12.4
cuDNN version: 90100


In [4]:
# appeler 
standardModelPath = "local_models\Roberta_model\models--xlm-roberta-base\snapshots\e73636d4f797dec63c3081bb6ed5c7b0bb3f2089"
device = torch.device("cuda")
tokenizer = XLMRobertaTokenizer.from_pretrained(standardModelPath)

<>:2: SyntaxWarning: invalid escape sequence '\R'
<>:2: SyntaxWarning: invalid escape sequence '\R'
C:\Users\Administrator\AppData\Local\Temp\ipykernel_15636\3420212622.py:2: SyntaxWarning: invalid escape sequence '\R'
  standardModelPath = "local_models\Roberta_model\models--xlm-roberta-base\snapshots\e73636d4f797dec63c3081bb6ed5c7b0bb3f2089"


In [26]:
complain_dataset = pandas.read_csv('DATA\MergedComplaints.csv')
complain_dataset = complain_dataset[ ['Commentaire client' , 'Predicted class' , 'Classe associé'] ]

# mapping
complain_dataset['Classe associé'] = complain_dataset['Classe associé'].map( {'Pannes et coupures générales':0 , 'Déploiement de la fibre':1 , 'Retards d’installation ou de mise en service':2 , 'Réclamation service':3 , 'Espace Client + My Idoom':4})
complain_dataset['Predicted class'] = complain_dataset['Predicted class'].map( {'Problème / Réclamation':0 , 'Inapproprié':1 , 'Question/Assistance':2 , 'Positif':3 , 'Suggestion':4})

# reonnmer en label 
complain_dataset = complain_dataset.rename(columns={
    'Predicted class': 'label_1',
    'Classe associé': 'label_2'}
    )

complain_dataset = complain_dataset.dropna(subset=['label_1'])
complain_dataset['label_2'] = complain_dataset['label_2'].fillna(-1)

# FORCE la conversion en entier demander par trainer
complain_dataset['label_1'] = complain_dataset['label_1'].astype(int)
complain_dataset['label_2'] = complain_dataset['label_2'].astype(int)

<>:1: SyntaxWarning: invalid escape sequence '\M'
<>:1: SyntaxWarning: invalid escape sequence '\M'
C:\Users\Administrator\AppData\Local\Temp\ipykernel_8732\2974156222.py:1: SyntaxWarning: invalid escape sequence '\M'
  complain_dataset = pandas.read_csv('DATA\MergedComplaints.csv')


In [27]:
complain_dataset['label_1'].value_counts()

label_1
0    18388
1     4426
2     3316
3     3072
4      769
Name: count, dtype: int64

In [28]:
complain_dataset['label_2'].value_counts()

label_2
-1    11583
 0     5106
 1     4606
 2     3510
 3     2870
 4     2296
Name: count, dtype: int64

In [29]:
#  preparer le metric et data 

from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix , f1_score
import gc

complain_train , complain_test = train_test_split( complain_dataset , test_size=0.2 , random_state=42 ,stratify=complain_dataset['label_2'] )
# stratiy  : pour garder les memes portions

# le dataset doit etre modifier car il manipulent plus que 1 label alors on herite de la class Dataset
class MultiTaskDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        text = str(self.data.iloc[index]['Commentaire client'])
        l1 = int(self.data.iloc[index]['label_1'])
        l2 = int(self.data.iloc[index]['label_2'])
    
        inputs = self.tokenizer(
            text,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_token_type_ids=False,
            return_tensors='pt'
        )

        return {
            # le text est compose de input id et attention mask
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'label_1': torch.tensor(l1, dtype=torch.long),
            'label_2': torch.tensor(l2, dtype=torch.long)
        }

# convertir en MultiTaskDataset
complain_trsain_Dataset = MultiTaskDataset(complain_train,tokenizer)
complain_tet_Dataset = MultiTaskDataset(complain_test,tokenizer)

# tout est pret on peut supp 
del complain_train , complain_test
gc.collect()

0

In [6]:
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    
    logits1, logits2 = logits
    labels1, labels2 = labels

    preds1 = logits1.argmax(-1)
    preds2 = logits2.argmax(-1)

    acc1 = accuracy_score(labels1, preds1)
    f1_1 = f1_score(labels1, preds1, average='weighted')
    cm1 = confusion_matrix(labels1, preds1)

    # TRES IMPORTANT 
    # On ne calcule que sur les lignes où label_2 != -1
    mask = (labels2 != -1)

    if mask.sum() > 0:
        # mapping
        acc2 = accuracy_score(labels2[mask], preds2[mask])
        f1_2 = f1_score(labels2[mask], preds2[mask], average='weighted')
        cm2 = confusion_matrix(labels2[mask], preds2[mask])
    else:
        acc2, f1_2 = 0, 0
        cm2 = None

    print("\n Matrice de Confusion 1 \n",cm1)
    print("\n Matrice de Confusion 2 \n",cm2)

    return {
        'accuracy_L1': acc1,
        'f1_L1': f1_1,
        'accuracy_L2': acc2,
        'f1_L2': f1_2,
    }

In [7]:
# cree le datacollator multitask
# features come from MultiTaskDataset returns
def Multi_DataCollator(features):
    # Prend une liste de sorties du Dataset et les transforme en un Batch
    batch = {
        'input_ids': torch.stack([f['input_ids'] for f in features]),
        'attention_mask': torch.stack([f['attention_mask'] for f in features]),
        'label_1': torch.stack([f['label_1'] for f in features]),
        'label_2': torch.stack([f['label_2'] for f in features])
    }
    
    return batch

In [5]:
# cree RobertaMultiTask
# code depuis gemini

class RobertaMultiTask(nn.Module):
    def __init__(self, model_path, num_labels_1, num_labels_2):
        super(RobertaMultiTask, self).__init__()
        self.roberta = XLMRobertaModel.from_pretrained(model_path)
        # Dropout pour éviter le surapprentissage
        self.dropout = nn.Dropout(0.1)
        
        # Tête 1 : Pour "Predicted class"
        self.classifier1 = nn.Linear(768, num_labels_1)
        
        # Tête 2 : Pour "Classe associé" , le null que jai eviter ne doit pas etre conte 
        self.classifier2 = nn.Linear(768, num_labels_2)

    def forward(self, input_ids, attention_mask, label_1=None, label_2=None):
        # Passage dans Roberta
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        
        # On récupère le token [CLS] (représentation globale)
        pooled_output = outputs.last_hidden_state[:, 0]
        pooled_output = self.dropout(pooled_output)
        
        # Calcul des logits pour chaque tête
        logits1 = self.classifier1(pooled_output)
        logits2 = self.classifier2(pooled_output)
        
        loss = None
        if label_1 is not None and label_2 is not None:
            # Definition du calcul de l'erreur (ignore_index=-1 pour le vide)
            loss_fct = nn.CrossEntropyLoss(ignore_index=-1)
            
            loss1 = loss_fct(logits1, label_1)
            loss2 = loss_fct(logits2, label_2)
            
            loss = 0.4*loss1 + 0.6*loss2
            
        # l'ordre est important        
        return {
        "loss": loss,
        "logits": (logits1, logits2) 
    }

In [9]:
# cree MultiTaskTrainer
class MultiTaskTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels_1 = inputs.get("label_1")
        labels_2 = inputs.get("label_2")
        
        # modèle retourne : loss, logits1, logits2
        outputs = model(
            input_ids=inputs.get("input_ids"),
            attention_mask=inputs.get("attention_mask"),
            label_1=labels_1,
            label_2=labels_2
        )
        
        # outputs[0] est le loss depuis RobertaMultiTask (model)
        loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        
        return (loss, outputs) if return_outputs else loss

In [ ]:
result_path = './RESULT_multiTask/'

training_args = TrainingArguments(
    output_dir= result_path + "MultiRoberta_finetuned",
    learning_rate=2e-5,  
    per_device_train_batch_size=16,  
    per_device_eval_batch_size=16,
    
    num_train_epochs=5, 
    weight_decay=0.01,
    
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True, 
    
    # choisit le 2eme car il est plus important et moins facile 
    metric_for_best_model="f1_L2", 
    greater_is_better=True,

    fp16=True, 
    dataloader_num_workers=0, 
    logging_steps=50,   

    # par défaut HuggingFace ne reconnaît pas daracolator personalisee
    remove_unused_columns=False,

)

giveModel = RobertaMultiTask(standardModelPath,5,5)
giveModel.to(device)

multi_trainer = MultiTaskTrainer(
    model= giveModel,
    args= training_args,
    train_dataset = complain_train_Dataset,
    eval_dataset = complain_test_Dataset,
    data_collator =  Multi_DataCollator,
    compute_metrics = compute_metrics
)

In [35]:
# start the training 
multi_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy L1,F1 L1,Accuracy L2,F1 L2
1,0.422142,0.341553,0.905254,0.901498,0.881185,0.879162
2,0.277239,0.293471,0.940117,0.938784,0.892605,0.888948
3,0.149873,0.196568,0.956297,0.955610,0.935563,0.935434
4,0.076022,0.182455,0.963303,0.962959,0.949701,0.949421
5,0.082079,0.171225,0.968307,0.968069,0.955682,0.955586



 Matrice de Confusion 1 
 [[3579   50   38    1   10]
 [ 185  639   24   14   13]
 [  78   17  603    0    1]
 [  16   33    0  540    2]
 [  54   14   12    6   66]]

 Matrice de Confusion 2 
 [[911  24  15  45  27]
 [ 10 865  36   7   3]
 [ 13  65 611  12   1]
 [ 88  35  36 404  11]
 [  2   1   0   6 450]]

 Matrice de Confusion 1 
 [[3571   69   21    4   13]
 [  72  786    4    8    5]
 [  39   19  639    0    2]
 [   3   23    0  564    1]
 [  34   26   13    3   76]]

 Matrice de Confusion 2 
 [[992  10  11   6   3]
 [ 22 849  47   2   1]
 [ 17  30 654   1   0]
 [151  16  47 355   5]
 [ 20   0   1   5 433]]

 Matrice de Confusion 1 
 [[3594   54   21    3    6]
 [  62  803    4    4    2]
 [  31    5  663    0    0]
 [   1   16    0  574    0]
 [  25   15    9    4   99]]

 Matrice de Confusion 2 
 [[969   4   8  33   8]
 [  5 876  32   7   1]
 [  9  33 648  11   1]
 [ 41  10  20 500   3]
 [  5   0   0   6 448]]

 Matrice de Confusion 1 
 [[3608   41   13    6   10]
 [  57  802 

TrainOutput(global_step=7495, training_loss=0.2753852200078678, metrics={'train_runtime': 756.4563, 'train_samples_per_second': 158.476, 'train_steps_per_second': 9.908, 'total_flos': 0.0, 'train_loss': 0.2753852200078678, 'epoch': 5.0})

In [36]:
# Sauvegarder le modele 

# Sauvegarder les poids du modèle
torch.save(multi_trainer.model.state_dict(), result_path+"MultiModel_classifier.pt")
tokenizer.save_pretrained(result_path+"tokenizer2_0")

print("Modèle multi task sauvegardé")

Modèle multi task sauvegardé


TEST

In [8]:
base_path = r"local_models/Roberta_model/models--xlm-roberta-base/snapshots/e73636d4f797dec63c3081bb6ed5c7b0bb3f2089"
model = RobertaMultiTask(base_path, num_labels_1=5, num_labels_2=5)

# Charger les poids sauvegardés
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.load_state_dict(torch.load(result_path + "MultiModel_classifier.pt", map_location=device))
model.to(device)
model.eval() 

# 2. Chargement du Tokenizer
tokenizer = XLMRobertaTokenizer.from_pretrained("RESULT_multiTask/tokenizer2_0")

# 3. Définition des mappings
id2label1 = {0:'Problème / Réclamation', 1:'Inapproprié', 2:'Question/Assistance', 3:'Positif', 4:'Suggestion'}
id2label2 = {None:'no sub-class asscociated', 0:'Pannes et coupures générales', 1:'Déploiement de la fibre', 2:'Retards d’installation ou de mise en service', 3:'Réclamation service', 4:'Espace Client + My Idoom'}

# 4. La fonction de prédiction
def predict(text):
    # Encodage du texte
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    # Envoyer les données sur le GPU
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits1, logits2 = outputs["logits"]

    # Récupération des index prédits
    p1_idx = logits1.argmax(-1).item()
    
    if p1_idx != 0:
        p2_idx = None
    else:
        p2_idx = logits2.argmax(-1).item()

    print(outputs["logits"])

    return id2label1[p1_idx], id2label2[p2_idx]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15463.66it/s]
XLMRobertaModel LOAD REPORT from: local_models/Roberta_model/models--xlm-roberta-base/snapshots/e73636d4f797dec63c3081bb6ed5c7b0bb3f2089
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
text = "الفيبر راه ميت قاع"
print([predict(text)])

(tensor([[ 7.2326, -1.3430, -2.5306, -2.1998, -2.3085]], device='cuda:0'), tensor([[-2.5922,  5.9984, -0.7830, -1.2024, -3.3224]], device='cuda:0'))
[('Problème / Réclamation', 'Déploiement de la fibre')]
